# YOLO para Sistema de Cámaras

Este notebook integra el modelo YOLO con el sistema de cámaras a través de la API.

In [14]:
import requests
import json
from datetime import datetime
import cv2
import numpy as np
from ultralytics import YOLO
import redis
import time
import os
from pathlib import Path
# Configuración desde variables de entorno
API_URL = os.getenv('API_URL', 'http://api-cam1:8000')
CAMERA_ID = os.getenv('CAMERA_ID', 'cam1')

print(f"Configuración para cámara {CAMERA_ID}")
print(f"Conectando a API: {API_URL}")

# Configuración de Redis
redis_client = redis.Redis(host='redis', port=6379, db=0)
print("Conectado a Redis")

# Directorios
VIDEOS_DIR = Path("videos")
FRAMES_DIR = Path("frames_analizados")
FRAMES_DIR.mkdir(exist_ok=True)
print(f"Directorio de videos: {VIDEOS_DIR}")
print(f"Directorio de frames: {FRAMES_DIR}")


Configuración para cámara cam1
Conectando a API: http://api-cam1:8000
Conectado a Redis
Directorio de videos: videos
Directorio de frames: frames_analizados


# Función para obtener videos pendientes

In [15]:
def get_next_video():
    """Obtiene el siguiente video pendiente de procesar"""
    print(f"Buscando videos pendientes para cámara {CAMERA_ID}")
    response = requests.get(f"{API_URL}/api/v1/videos")
    if response.status_code == 200:
        videos = response.json()
        # Filtrar videos pendientes de esta cámara
        pending_videos = [v for v in videos if v['status'] == 'pending' and v['camera_id'] == CAMERA_ID]
        if pending_videos:
            print(f"Video encontrado: {pending_videos[0]['filename']}")
            return pending_videos[0]
        else:
            print("No hay videos pendientes")
    else:
        print(f"Error al obtener videos: {response.status_code}")
    return None

# Función para procesar video

In [16]:
def process_video(video_data):
    """Procesa un video con YOLO y envía resultados a Redis"""
    print(f"Iniciando procesamiento de {video_data['filename']}")
    video_path = VIDEOS_DIR / video_data['filename']
    cap = cv2.VideoCapture(str(video_path))

    # Inicializar YOLO
    print("Cargando modelo YOLO...")
    model = YOLO('yolov8m.pt')
    print("Modelo YOLO cargado")

    frame_number = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Total de frames a procesar: {total_frames}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Procesar frame con YOLO
        results = model(frame, conf=0.5)

        # Preparar detecciones
        detections = []
        for result in results:
            boxes = result.boxes
            for box in boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                if cls == 2:  # Solo vehículos
                    detections.append({
                        'confidence': conf,
                        'bbox': box.xyxy[0].tolist()
                    })

        # Publicar resultados en Redis
        message = {
            'frame_number': frame_number,
            'timestamp': datetime.now().isoformat(),
            'camera_id': CAMERA_ID,
            'detections': detections
        }

        redis_client.publish(
            f'camera_{CAMERA_ID}',
            json.dumps(message)
        )

        if frame_number % 100 == 0:  # Mostrar progreso cada 100 frames
            print(f"Progreso: {frame_number}/{total_frames} frames procesados")

        frame_number += 1

    cap.release()
    print(f"Procesamiento completado: {frame_number} frames procesados")

    # Actualizar estado del video
    print("Actualizando estado del video...")
    requests.post(
        f"{API_URL}/api/v1/videos/status/{video_data['filename']}",
        json={'status': 'completed'}
    )
    print("Estado actualizado")

# Bucle principal

In [17]:
def main_loop():
    """Bucle principal que procesa videos pendientes"""
    print(f"Iniciando procesamiento para cámara {CAMERA_ID}")
    while True:
        video_data = get_next_video()
        if video_data:
            print(f"Procesando video: {video_data['filename']}")
            process_video(video_data)
        else:
            print("No hay videos pendientes, esperando...")
            time.sleep(5)

## Iniciar procesamiento

In [18]:
if __name__ == "__main__":
    main_loop()

Iniciando procesamiento para cámara cam1
Buscando videos pendientes para cámara cam1


ConnectionError: HTTPConnectionPool(host='api-cam1', port=8000): Max retries exceeded with url: /api/v1/videos (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x0000020B8D21DF90>: Failed to resolve 'api-cam1' ([Errno 11001] getaddrinfo failed)"))

## Prueba del Sistema